# Model 7 · ambiguity-aware gallery pairs
Starting point: Dalvin’s Model 3d. Sani, Khurana and Anand select ambiguous pairs using disagreement between DBSCAN and FINCH, request identity constraints, and refine clusters. Here first-level FINCH and DBSCAN select difficult, already-labelled gallery pairs for a logistic same-animal classifier. Gallery identities supply pair labels. Competition-query identities are never requested. This adaptation omits the paper’s annotation loop and neural training. It also adjusts SIFT evidence using each gallery image’s verified different-animal matches. [AAAI paper](https://ojs.aaai.org/index.php/AAAI/article/view/41266)

In [1]:
import os
import re
from multiprocessing import Pool

import cv2
import lightgbm as lgb
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.sparse.csgraph import connected_components
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA


from pathlib import Path
from sklearn.linear_model import Ridge
from threadpoolctl import threadpool_limits
from hashlib import sha256
FULL_RUN = True
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'Processed_Data/master_dataset.csv').exists())
processed = root/'Processed_Data'
meta = pd.read_csv(processed/'master_dataset.csv')
assert meta.id.is_unique and set(meta.split) == {'train','test'}
train = meta[meta.split=='train'].reset_index(drop=True)
test = meta[meta.split=='test'].reset_index(drop=True)
rows = pd.concat([train,test], ignore_index=True)
features = np.load(processed/'miewid_features.npy', mmap_mode='r')
idx = rows.miewid_index.to_numpy(int)
assert np.array_equal(np.sort(idx), np.arange(len(features)))
X = features[idx].copy()
X /= np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-12)
IDS, SP, Y = rows.id.to_numpy(), rows.species.to_numpy(), rows.identity.to_numpy(object)
NTR = len(train)
DATA = str(root/'DATA/reid_student_data_main')
OUT = str(root/'Models/Emad'/('cross_flank_' + sha256((' '.join(IDS)).encode()).hexdigest()[:12]))
N_SIMS, FOLDS, ROUNDS = 32, 8, 3        # simulated splits, stacking folds, stacking rounds
K, NEIGHBOURS = 15, 150                 # candidates per query, SIFT pairs verified per image
MATCH_RATIO, RANSAC_PX = 0.85, 15.0         # SIFT matching: ratio test, RANSAC pixel tolerance
SIFT_PULL = 0.05                        # weight of SIFT evidence when selecting candidates
FLANK = np.array([0, 1, 2, 2, 0, 1, 0, 1, 2, 2])    # left-ish, right-ish, other (per VIEWS order)
VIEWS = ["left", "right", "front", "back", "frontleft", "frontright", "backleft", "backright", "up", "down"]


VIEW = rows.viewpoint.map(dict(zip(VIEWS, range(len(VIEWS))))).fillna(-1).to_numpy(int)
side = np.where(VIEW >= 0, FLANK[VIEW.clip(0)], 2)
print(len(train), 'gallery images;', len(test), 'competition queries')

7163 gallery images; 2791 competition queries


## Gallery-only pair model
Within each species, DBSCAN uses the 60th percentile of third-neighbour distances as its radius. First-level FINCH connects each image to its nearest neighbour (shared-neighbour links are implicit in the connected components). Each gallery anchor contributes up to three disagreement pairs plus three uniformly sampled controls from its twelve nearest neighbours. The logistic model uses cosine, inlier count, inlier excess above the gallery image’s usual false match, and view agreement. The false-match estimate shrinks toward the species mean with five prior observations. Leave the candidate shortlist and original ranker inputs intact.

In [2]:

from sklearn.cluster import DBSCAN
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def pair_features(gi, qi, raw, inliers, inv, candidates):
    g= X[gi]
    dist=np.maximum(0,1-g@g.T)
    np.fill_diagonal(dist,np.inf)
    neighbors=np.argsort(dist,axis=1)[:,:12]
    radius=max(1e-6,float(np.quantile(dist[np.arange(len(g)),neighbors[:,2]],.6)))
    np.fill_diagonal(dist,0)
    density=DBSCAN(eps=radius,min_samples=4,metric='precomputed').fit_predict(dist)
    rows=np.arange(len(g))
    graph=sparse.csr_matrix((np.ones(len(g)),(rows,neighbors[:,0])),shape=dist.shape)
    first=connected_components(graph,directed=False)[1]
    edges=VERIFIED[gi][:,gi].tocoo()
    a,b=edges.row,edges.col
    keep=inv[a]!=inv[b]
    a,b=a[keep],b[keep]
    within=INL[gi][:,gi].toarray()
    hits=within[a,b]
    prior=hits.mean() if len(hits) else 0.
    background=(np.bincount(b,weights=hits,minlength=len(g))+5*prior)/(np.bincount(b,minlength=len(g))+5)
    rng=np.random.default_rng(918)
    pairs=set()
    for i,nbh in enumerate(neighbors):
        agree_density=(density[i]>=0)&(density[nbh]==density[i])
        hard=nbh[agree_density!=(first[i]==first[nbh])]
        chosen=np.r_[rng.permutation(hard)[:3],rng.choice(nbh,3,replace=False)]
        pairs.update((i,j) for j in chosen)
    a,b=np.asarray(sorted(pairs)).T
    def design(sim,hits,b,view):
        return np.column_stack([sim,np.log1p(hits),np.log1p(hits)-np.log1p(background[b]),view])
    def same_view(x,y):
        return (x>=0)&(x==y)
    labels=inv[a]==inv[b]
    if len(np.unique(labels))<2:
        return np.full(candidates.size,np.nan),np.full(candidates.size,np.nan)
    clf=make_pipeline(StandardScaler(),LogisticRegression(C=1,max_iter=300))
    clf.fit(design(1-dist[a,b],within[a,b],b,same_view(VIEW[gi[a]],VIEW[gi[b]])),labels)
    scores=np.empty(candidates.shape)
    excess=np.empty(candidates.shape)
    for ident in np.unique(candidates):
        r,c=np.where(candidates==ident)
        cols=np.flatnonzero(inv==ident)
        qidx,b=np.broadcast_arrays(r[:,None],cols)
        flatq,b=qidx.ravel(),b.ravel()
        hits=inliers[flatq,b]
        feats=design(raw[flatq,b],hits,b,same_view(VIEW[qi[flatq]],VIEW[gi[b]]))
        scores[r,c]=clf.decision_function(feats).reshape(qidx.shape).max(1)
        excess[r,c]=feats[:,2].reshape(qidx.shape).max(1)
    return scores.ravel(),excess.ravel()


In [3]:
def load_sift():
    directory = processed/'sift_features'
    xy, descriptors = [], []
    for ident in IDS:
        xy.append(np.load(directory/f'{ident}_xy.npy'))
        d = np.load(directory/f'{ident}_desc.npy').astype(np.float32)
        descriptors.append(np.sqrt(d / np.maximum(d.sum(1,keepdims=True),1)))
    return xy, descriptors
def verify(pair):
    """(mutual ratio-test matches, RANSAC inliers) between two images."""
    a, b = pair
    if len(DESC[a]) < 4 or len(DESC[b]) < 4:
        return 0, 0
    bf = cv2.BFMatcher()
    back = np.array([m.trainIdx for m in bf.match(DESC[b], DESC[a])])           # nearest descriptor of b in a
    good = [m[0] for m in bf.knnMatch(DESC[a], DESC[b], k=2)
            if len(m) == 2 and m[0].distance < MATCH_RATIO * m[1].distance and back[m[0].trainIdx] == m[0].queryIdx]
    if len(good) < 4:
        return len(good), 0
    src = np.float32([XY[a][m.queryIdx] for m in good])
    dst = np.float32([XY[b][m.trainIdx] for m in good])
    _, inliers = cv2.estimateAffinePartial2D(src, dst, method=cv2.RANSAC, ransacReprojThreshold=RANSAC_PX)
    return len(good), 0 if inliers is None else int(inliers.sum())


def sift_matrices():
    """Symmetric sparse matrices of matches / inliers for every image and its nearest same-species neighbours.
    Pairs missing from the cache are verified in parallel and appended."""
    path, n = f"{OUT}/sift_pairs_mutual.npz", len(X)
    wanted = set()
    for sp in ("hyena", "leopard"):
        ix = np.where(SP == sp)[0]
        sim = X[ix] @ X[ix].T
        np.fill_diagonal(sim, -9)
        for row, nb in zip(ix, np.argsort(-sim, 1)[:, :NEIGHBOURS]):
            wanted.update(min(row, j) * n + max(row, j) for j in ix[nb])
    done = np.load(path) if os.path.exists(path) else dict(pairs=np.empty((0, 2), int), good=np.empty(0, int), inl=np.empty(0, int))
    missing = np.setdiff1d(np.fromiter(wanted, np.int64), done["pairs"][:, 0] * n + done["pairs"][:, 1])
    if len(missing):
        global XY, DESC
        XY, DESC = load_sift()
        new = np.stack(divmod(missing, n), 1)
        with Pool(min(16, len(os.sched_getaffinity(0)))) as pool:
            res = np.empty((len(new),2), int)
            print(f"Computing {len(new)} SIFT pairs", flush=True)
            for i, counts in enumerate(pool.imap(verify, map(tuple, new), chunksize=128)):
                res[i] = counts
                if (i+1)%100000 == 0:
                    print(f"Verified {i+1}/{len(new)}", flush=True)
        done = dict(pairs=np.r_[done["pairs"], new], good=np.r_[done["good"], res[:, 0]], inl=np.r_[done["inl"], res[:, 1]])
        os.makedirs(OUT, exist_ok=True)
        np.savez(path, **done)
    a, b = done["pairs"].T
    mat = lambda v: sparse.coo_matrix((np.r_[v, v], (np.r_[a, b], np.r_[b, a])), shape=(n, n)).tocsr()
    return mat(done["good"]), mat(done["inl"])

In [4]:
def unit(a):
    return a / np.linalg.norm(a, axis=1, keepdims=True)


def embed(gallery, labels, queries):
    pca = PCA(min(384, len(gallery) + len(queries) - 1)).fit(np.r_[gallery, queries])      # unsupervised, sees the queries too
    lda = LDA(solver="eigen", shrinkage=0.3).fit(pca.transform(gallery), labels)
    project = lambda a: unit(lda.transform(pca.transform(a)))
    return project(gallery), project(queries)


def per_identity(sim, inv, n):
    """Per query: max and mean-of-top-2 similarity to each identity's gallery images."""
    best, top2 = np.empty((sim.shape[0], n)), np.empty((sim.shape[0], n))
    for j in range(n):
        s = sim[:, inv == j]
        best[:, j], top2[:, j] = s.max(1), np.sort(s, 1)[:, -2:].mean(1)
    return best, top2


def species_features(gi, qi, sp):
    """Query frame and (query, candidate) frame for one species; gi/qi are global image indices."""
    Rg, Rq, yg, vg, vq = X[gi], X[qi], Y[gi], VIEW[gi], VIEW[qi]
    Lg, Lq = embed(Rg, yg, Rq)
    ids, inv = np.unique(yg, return_inverse=True)
    n, nq = len(ids), len(qi)
    sim_l, sim_r = Lq @ Lg.T, Rq @ Rg.T
    lda1, lda2 = per_identity(sim_l, inv, n)
    raw1, raw2 = per_identity(sim_r, inv, n)
    centroid = unit(np.stack([Lg[inv == j].mean(0) for j in range(n)]))
    cen = Lq @ centroid.T
    inl_g, good_g = INL[qi][:, gi].toarray().astype(np.float32), GOOD[qi][:, gi].toarray().astype(np.float32)
    sift1, sift2 = per_identity(inl_g, inv, n)
    good1, _ = per_identity(good_g, inv, n)
    support = (inl_g >= 7).astype(np.float32) @ np.eye(n, dtype=np.float32)[inv]     # gallery images confirmed per identity
    same_view = np.full((nq, n), np.nan)
    for j in range(n):
        for v in np.unique(vq):
            cols = np.where((inv == j) & (vg == v))[0]
            if len(cols):
                same_view[vq == v, j] = sim_l[np.ix_(vq == v, cols)].max(1)
    # viewpoint coverage: flanks of one animal look different, so unseen sides cannot be matched
    by_view = np.stack([np.bincount(inv[vg == v], minlength=n) for v in range(len(VIEWS))], 1)
    by_flank = np.stack([by_view[:, FLANK == f].sum(1) for f in range(3)], 1)
    seen = vq >= 0
    n_view = np.where(seen[:, None], by_view[:, vq.clip(0)].T, np.nan)
    n_flank = np.where(seen[:, None], by_flank[:, FLANK[vq.clip(0)]].T, np.nan)
    # transductive: neighbours among the queries themselves
    qq_r, qq_l, qq_s = Rq @ Rq.T, Lq @ Lq.T, INL[qi][:, qi].toarray().astype(np.float32)
    np.fill_diagonal(qq_r, -1), np.fill_diagonal(qq_l, -1)
    nb, sb = np.argsort(-qq_l, 1)[:, :5], np.argsort(-qq_s, 1)[:, :5]
    nbs, sbs = np.take_along_axis(qq_l, nb, 1), np.take_along_axis(qq_s, sb, 1)
    t_l = np.take_along_axis(lda1, np.argsort(-lda1, 1)[:, :3], 1)
    t_r = -np.sort(-raw1, 1)[:, :3]
    t_s = -np.sort(-sift1, 1)[:, :3]
    g_top = -np.sort(-sim_l, 1)
    q = pd.DataFrame({
        "sp": int(sp == "hyena"), "view": vq, "ng": len(gi),
        "l1": t_l[:, 0], "l2": t_l[:, 1], "l3": t_l[:, 2], "r1": t_r[:, 0], "r2": t_r[:, 1], "r3": t_r[:, 2],
        "lm12": t_l[:, 0] - t_l[:, 1], "rm12": t_r[:, 0] - t_r[:, 1],
        "gd5": g_top[:, :5].mean(1), "gd20": g_top[:, :20].mean(1),
        "qq1": nbs[:, 0], "qq3": nbs[:, :3].mean(1), "qq5": nbs.mean(1), "qqr1": qq_r.max(1),
        "qn6": (qq_r > .6).sum(1), "qn7": (qq_r > .7).sum(1), "qn8": (qq_r > .8).sum(1),
        "qq_minus_g": nbs[:, 0] - t_l[:, 0], "nb_g1": t_l[nb[:, 0], 0],
        "s1": t_s[:, 0], "s2": t_s[:, 1], "s3": t_s[:, 2], "sg": good_g.max(1),
        "qs1": sbs[:, 0], "qs3": sbs[:, :3].mean(1), "qsn10": (qq_s >= 10).sum(1), "qsn20": (qq_s >= 20).sum(1),
        "qs_minus_s": sbs[:, 0] - t_s[:, 0], "nbs_s1": t_s[sb[:, 0], 0],
    })
    for k in range(5):
        q[f"nb{k}"], q[f"nw{k}"], q[f"sb{k}"], q[f"sw{k}"] = nb[:, k], nbs[:, k], sb[:, k], sbs[:, k]
    # candidates: best identities by embedding similarity, lightly boosted by SIFT evidence
    cand = np.argsort(-(lda1 + SIFT_PULL * np.tanh(sift1 / 15)), 1)[:, :K]
    r, c = np.repeat(np.arange(nq), K), cand.ravel()
    nb_ev = np.max(np.minimum(nbs[:, :, None], np.stack([lda1[nb[:, k]] for k in range(5)], 1)), 1)
    sb_ev = np.max(np.minimum(sbs[:, :, None], np.stack([sift1[sb[:, k]] for k in range(5)], 1)), 1)
    rank_of = lambda m: (-m).argsort(1).argsort(1)[r, c]
    cd = pd.DataFrame({
        "q": r, "c": c, "rank": np.tile(np.arange(K), nq), "ident": ids[c],
        "mxr": raw1[r, c], "m2r": raw2[r, c], "mxl": lda1[r, c], "m2l": lda2[r, c], "cl": cen[r, c],
        "gapl": lda1[r, c] - t_l[r, 0], "gapr": raw1[r, c] - t_r[r, 0], "same": same_view[r, c],
        "cnt": np.bincount(inv)[c], "nbev": nb_ev[r, c], "sbev": sb_ev[r, c],
        "rk_r": rank_of(raw1), "rk_c": rank_of(cen), "rk_s": rank_of(sift1),
        "n_view": n_view[r, c], "n_flank": n_flank[r, c], "support": support[r, c], "sx": sift1[r, c], "s2x": sift2[r, c], "gx": good1[r, c], "sgap": sift1[r, c] - t_s[r, 0],
    })
    if use_extension:
        cd['pair_score'],cd['sift_excess']=pair_features(gi,qi,sim_r,inl_g,inv,cand)
    return q, cd


def build_frames(gi, qi):
    qs, cs, order, off = [], [], [], 0
    for sp in ("hyena", "leopard"):
        g, qq = gi[SP[gi] == sp], qi[SP[qi] == sp]
        q, c = species_features(g, qq, sp)
        c["q"] += off
        q[[f"{p}{k}" for p in ("nb", "sb") for k in range(5)]] += off
        qs.append(q), cs.append(c), order.append(qq)
        off += len(qq)
    return dict(q=pd.concat(qs, ignore_index=True), c=pd.concat(cs, ignore_index=True), order=np.concatenate(order))


# ----------------------------------------------------------------------------- open-set simulation
# Near-duplicate linking (raw similarity, SIFT inliers) per regime; None = random image holdout.
REGIMES = [None, (0.95, 111), (0.99, 89), (0.9, 67)]


def duplicate_groups(ix, link):
    """Group images of one identity that are near-duplicates (same encounter); each image alone if link is None."""
    if link is None:
        return np.arange(len(ix))
    edges = (X[ix] @ X[ix].T > link[0]) | (INL[ix][:, ix].toarray() >= link[1])
    return connected_components(sparse.csr_matrix(edges))[1]


def simulate(seed):
    """Identity-disjoint open-set split mimicking the test set: unknown identities, capped queries per
    identity, and (in most regimes) whole near-duplicate groups held out together."""
    rng, link = np.random.default_rng(seed), REGIMES[seed % len(REGIMES)]
    gallery, known, unknown = [], [], []
    for sp in ("hyena", "leopard"):
        p_unknown, frac, cap = rng.uniform(.2, .35), rng.uniform(.3, .5), int(rng.integers(12, 33))
        d = train[train.species == sp]
        names = np.array(sorted(d.identity.unique()))
        rng.shuffle(names)
        new = set(names[:int(round(p_unknown * len(names)))])
        for name, g in d.groupby("identity"):
            ix = g.index.to_numpy().copy()
            rng.shuffle(ix)
            if name in new:
                unknown += list(ix[:cap])
            elif len(ix) < 2:
                gallery += list(ix)
            else:
                n_q, held = min(max(int(round(frac * len(ix))), 1), cap, len(ix) - 1), []
                groups = duplicate_groups(ix, link)
                for c in rng.permutation(np.unique(groups)):
                    members = list(ix[groups == c])
                    if len(held) < n_q and len(held) + len(members) < len(ix):
                        held += members
                known += held[:cap]
                gallery += [i for i in ix if i not in set(held)]
    frames = build_frames(np.array(gallery), np.array(known + unknown))
    truth = np.where(np.isin(frames["order"], unknown), "new_individual", Y[frames["order"]])
    frames["truth"] = truth
    frames["q"]["unk"] = (truth == "new_individual").astype(int)
    frames["c"]["y"] = (frames["c"].ident.to_numpy(dtype=object) == truth[frames["c"].q.to_numpy()]).astype(int)
    return frames


# ----------------------------------------------------------------------------- stacked LightGBM
PARAMS = dict(objective="binary", learning_rate=.05, num_leaves=31, min_child_samples=30, subsample=.8,
              subsample_freq=1, colsample_bytree=.8, n_estimators=400, verbose=-1)
INDEX_COLS = re.compile(r"^(nb|nw|sb|sw)\d$")


def columns(s):
    qf = [c for c in s["q"].columns if c != "unk" and not INDEX_COLS.match(c)]
    cf = [c for c in s["c"].columns if c not in ("q", "c", "ident", "y")]
    return qf, cf


def candidate_matrix(s, qf, cf):
    return pd.concat([s["c"][cf].reset_index(drop=True), s["q"].iloc[s["c"].q.to_numpy()][qf].reset_index(drop=True)], axis=1)


def fit(sims):
    """new-individual detector, closed-set candidate scorer, and all-query candidate scorer."""
    qf, cf = columns(sims[0])
    known = lambda s: s["q"].unk.to_numpy()[s["c"].q.to_numpy()] == 0
    cx = pd.concat([candidate_matrix(s, qf, cf) for s in sims])
    cy = np.concatenate([s["c"].y.to_numpy() for s in sims])
    ck = np.concatenate([known(s) for s in sims])
    detector = lgb.LGBMClassifier(**{**PARAMS, "n_estimators": 300, "num_leaves": 15}).fit(
        pd.concat([s["q"][qf] for s in sims]), np.concatenate([s["q"].unk.to_numpy() for s in sims]))
    return detector, lgb.LGBMClassifier(**PARAMS).fit(cx[ck], cy[ck]), lgb.LGBMClassifier(**PARAMS).fit(cx, cy), (qf, cf)


def predict(models, s):
    detector, closed, anyq, (qf, cf) = models
    nq, cx = len(s["q"]), candidate_matrix(s, qf, cf)
    p_new = detector.predict_proba(s["q"][qf])[:, 1]
    raw = anyq.predict_proba(cx)[:, 1].reshape(nq, K)
    within = closed.predict_proba(cx)[:, 1].reshape(nq, K)
    within /= np.maximum(within.sum(1, keepdims=True), 1e-9)
    return dict(pn=p_new, raw=raw, pc=within, pb=(p_new + np.clip(1 - raw.sum(1), 0, 1)) / 2)


def rank(s, r):
    """Top-5 labels: P(new) vs (1 - P(new)) * P(identity | known)."""
    nq = len(s["q"])
    labels = np.concatenate([s["c"].ident.to_numpy(dtype=object).reshape(nq, K), np.full((nq, 1), "new_individual", dtype=object)], 1)
    scores = np.concatenate([(1 - r["pb"])[:, None] * r["pc"], r["pb"][:, None]], 1)
    return np.take_along_axis(labels, np.argsort(-scores, 1)[:, :5], 1)


def map5(preds, truth):
    return np.mean([next((1 / (i + 1) for i, p in enumerate(row) if p == t), 0.0) for row, t in zip(preds, truth)])


def augment(s, r):
    """Add the previous round's predictions for each query and for its nearest query neighbours."""
    q, c, nq = s["q"].copy(), s["c"].copy(), len(s["q"])
    codes = pd.factorize(c.ident)[0].reshape(nq, K)
    q["r_pn"], q["r_pb"], q["r_rawsum"], q["r_rawmax"] = r["pn"], r["pb"], r["raw"].sum(1), r["raw"].max(1)
    c["r_raw"], c["r_pc"] = r["raw"].ravel(), r["pc"].ravel()
    for kind, weight, scale in (("nb", "nw", 1.0), ("sb", "sw", 30.0)):     # embedding / SIFT neighbours
        evidence, support, pb, w, top = np.zeros((nq, K)), np.zeros((nq, K)), [], [], []
        for k in range(5):
            idx, sim = q[f"{kind}{k}"].to_numpy(), q[f"{weight}{k}"].to_numpy() / scale
            wk = np.clip(sim - 0.4, 0, None) if kind == "nb" else np.clip(sim, 0, 1)
            agree = ((codes[idx][:, None, :] == codes[:, :, None]) * r["raw"][idx][:, None, :]).sum(2)
            evidence = np.maximum(evidence, np.minimum(agree, np.clip(sim, 0, 1)[:, None]))
            support += wk[:, None] * agree
            pb.append(r["pb"][idx]), w.append(wk), top.append(r["raw"][idx].max(1))
        pb, w = np.stack(pb, 1), np.stack(w, 1)
        q[f"{kind}_pb0"], q[f"{kind}_pbw"] = pb[:, 0], (pb * w).sum(1) / np.maximum(w.sum(1), 1e-6)
        q[f"{kind}_pbmin"], q[f"{kind}_w"], q[f"{kind}_rm0"] = np.where(w > 0, pb, 1).min(1), w.sum(1), top[0]
        c[f"r_{kind}ev"], c[f"r_{kind}sum"] = evidence.ravel(), support.ravel()
    return dict(s, q=q, c=c)

## Identity-disjoint open-set pilot
Per species, draw up to 100 animals with at least two photos and up to eight photos per animal. Half the animals supply scorer training episodes and half supply evaluation. Limit mutual SIFT verification to three nearest neighbours within each role and species. Compare first-round scorers on the same held-out episode. This small-gallery experiment trades coverage and stacking depth for a repeatable local comparison.

In [5]:
def run_pilot():
    global train, INL, GOOD, DESC, XY, use_extension, VERIFIED
    original = train
    rng = np.random.default_rng(3291)
    roles = [[], []]
    for _, group in original.groupby('species'):
        counts = group.identity.value_counts()
        names = rng.permutation(counts[counts >= 2].index)[:100]
        for role, animals in zip(roles, np.array_split(names, 2)):
            for animal in animals:
                role.extend(rng.permutation(group.index[group.identity == animal])[:8])
    pairs = set()
    for role in roles:
        for sp in np.unique(SP):
            ix = np.array([j for j in role if SP[j] == sp])
            sim = X[ix] @ X[ix].T
            np.fill_diagonal(sim, -np.inf)
            for a, neighbors in zip(ix, ix[np.argsort(-sim, axis=1)[:, :3]]):
                pairs.update(tuple(sorted((a,b))) for b in neighbors)
    XY, DESC = {}, {}
    for j in roles[0] + roles[1]:
        prefix = processed/'sift_features'/IDS[j]
        XY[j] = np.load(str(prefix)+'_xy.npy')
        d = np.load(str(prefix)+'_desc.npy').astype(np.float32)
        DESC[j] = np.sqrt(d / np.maximum(d.sum(1,keepdims=True),1))
    pairs = sorted(pairs)
    a,b = np.array(pairs).T
    good,inliers = np.array([verify(pair) for pair in pairs]).T
    def matrix(v):
        return sparse.csr_matrix((np.r_[v,v],(np.r_[a,b],np.r_[b,a])),shape=(len(X),len(X)))
    GOOD, INL = matrix(good),matrix(inliers)
    VERIFIED = matrix(np.ones(len(a), dtype=np.int8))
    use_extension = True
    try:
        train = original.loc[roles[0]]
        episodes = [simulate(seed) for seed in (11,12,13,14)]
        train = original.loc[roles[1]]
        held = simulate(2026)
        for label in ('3d + both', '3d + excess', '3d'):
            if label != '3d + both':
                for frame in episodes + [held]:
                    frame['c'] = frame['c'].drop(columns='sift_excess' if label=='3d' else 'pair_score')
            pred = rank(held,predict(fit(episodes),held))
            truth = held['truth']
            known = truth != 'new_individual'
            print(label, len(truth), 'queries; overall / known / unknown MAP@5',
                  *(round(map5(pred[m],truth[m]),5) for m in (np.ones_like(known),known,~known)))
    finally:
        train = original
        DESC.clear()
        XY.clear()

cv2.setNumThreads(1)
PARAMS['n_jobs'] = 4
with threadpool_limits(limits=4):
    if not FULL_RUN:
        run_pilot()

## Full competition pipeline
Enable FULL_RUN to verify 150 neighbours per image and run the three-round 3d stack with and without the extension. Each variant produces a local CSV. Full mode is enabled. Disable FULL_RUN to run the identity-disjoint pilot.

In [6]:
def run_stack(variant):
    np.random.seed(2026)
    print(f"simulating {N_SIMS} open-set splits...")
    sims = [simulate(seed) for seed in range(N_SIMS)]
    final = build_frames(np.arange(NTR), np.arange(NTR, len(X)))
    folds = [list(range(i, N_SIMS, FOLDS)) for i in range(FOLDS)]
    for rnd in range(1, ROUNDS + 1):
        oof = [None] * N_SIMS
        for fold_index,f in enumerate(folds):
            print(variant, "round", rnd, "fold", fold_index+1, flush=True)
            models = fit([sims[i] for i in range(N_SIMS) if i not in f])
            for i in f:
                oof[i] = predict(models, sims[i])
        score = np.mean([map5(rank(sims[i], oof[i]), sims[i]["truth"]) for i in range(N_SIMS)])
        print(f"round {rnd}: out-of-fold simulated MAP@5 = {score:.4f}")
        result = predict(fit(sims), final)
        if rnd < ROUNDS:
            sims, final = [augment(s, r) for s, r in zip(sims, oof)], augment(final, result)
    preds = rank(final, result)
    sub = pd.DataFrame({"id": IDS[final["order"]], "predictions": [" ".join(p) for p in preds]}).sort_values("id")
    sample = pd.read_csv(f"{DATA}/sample_submission.csv")
    assert set(sub.id) == set(sample.id) and sub.id.is_unique and (sub.predictions.str.split().map(len) == 5).all()
    os.makedirs(OUT, exist_ok=True)
    sub = sample[["id"]].merge(sub, on="id", validate="one_to_one")
    assert sub.predictions.map(lambda x: len(set(x.split())) == 5).all()
    sub.to_csv(f"{OUT}/submission_{variant}.csv", index=False)
    print(f"saved {len(sub)} rows; new_individual ranked first for {np.mean(preds[:, 0] == 'new_individual'):.1%}")

    return score

if FULL_RUN:
    os.makedirs(OUT, exist_ok=True)
    cv2.setNumThreads(1)
    with threadpool_limits(limits=4):
        GOOD, INL = sift_matrices()
        pairs=np.load(Path(OUT)/'sift_pairs_mutual.npz')['pairs']
        a,b=pairs.T
        VERIFIED=sparse.csr_matrix((np.ones(2*len(a),np.int8),(np.r_[a,b],np.r_[b,a])),shape=GOOD.shape)
        for use_extension, variant in [(False,'3d'),(True,'model7_ambiguity')]:
            print(variant, run_stack(variant))

Computing 1047159 SIFT pairs


Verified 100000/1047159


Verified 200000/1047159


Verified 300000/1047159


Verified 400000/1047159


Verified 500000/1047159


... (192 outputs left out)

model7_ambiguity round 3 fold 7


model7_ambiguity round 3 fold 8


round 3: out-of-fold simulated MAP@5 = 0.9130


saved 2791 rows; new_individual ranked first for 40.6%
model7_ambiguity 0.913008497008539


## Competition result

Model 7 scored **0.85187** on the public Kaggle leaderboard, 0.00773 below Model 3d (0.85960). Full local simulated MAP@5 was 0.91301 for Model 7 and 0.91249 for the matched 3d control.